# R3 — Window Functions + Cohorts (w01..w04)

Binding definitions (grader uses exactly these):

* **Paid** — approved amount of **settled** claims, grouped by the month the claim actually **settled** in and by the plan **at admission**.
* **Earned premium** — one-twelfth of the annual premium for every plan version active on **any day of the month**; a version active for part of a month still earns a **full** month's premium (simplification, not proration).
* **Loss ratio** — paid ÷ earned premium (loss ratios here are far above 1 because the sample book is small — expected, do not "fix").
* **Rolling 3-month loss ratio** — a **single ratio** over the plan's current month + the two months before (sum paid ÷ sum earned over the 3-month window), **not** three monthly ratios averaged.
* **Rank** — hospitals genuinely tied in a city tier **share** a rank (`RANK()`, not `ROW_NUMBER()`).
* **Cohort** — members grouped by the quarter they **started their policy**; claim activity of that group over time.

In [0]:
%sql
-- w01 materialized as a table (the delivery contract ships claims.analytics.w01) —
-- it is also the source R4's mv_loss_ratio reads from.
CREATE OR REPLACE TABLE claims.analytics.w01 AS
WITH months AS (
  SELECT DISTINCT
    month_start,
    ADD_MONTHS(month_start, 1) - INTERVAL 1 DAY AS month_end
  FROM claims.analytics.dim_date
),
paid AS (
  SELECT
    p.plan_code,
    d.month_start,
    SUM(f.approved_amount) AS paid_amount
  FROM claims.analytics.fact_claims f
  JOIN claims.analytics.dim_plan p ON f.plan_sk = p.plan_sk
  JOIN claims.analytics.dim_date d ON f.settled_date_key = d.date_key
  WHERE f.status = 'settled'
  GROUP BY p.plan_code, d.month_start
),
premium AS (
  -- a plan version earns a full month's premium if it was active on ANY day of that month
  SELECT
    p.plan_code,
    m.month_start,
    SUM(p.annual_premium) / 12.0 AS earned_premium
  FROM claims.analytics.dim_plan p
  JOIN months m
    ON p.valid_from <= m.month_end
   AND (p.valid_to IS NULL OR p.valid_to >= m.month_start)
  GROUP BY p.plan_code, m.month_start
)
SELECT
  COALESCE(pr.plan_code, pd.plan_code) AS plan_code,
  COALESCE(pr.month_start, pd.month_start) AS month_start,
  ROUND(COALESCE(pd.paid_amount, 0), 2)    AS paid_amount,
  ROUND(COALESCE(pr.earned_premium, 0), 2) AS earned_premium,
  ROUND(COALESCE(pd.paid_amount, 0) / NULLIF(COALESCE(pr.earned_premium, 0), 0), 2) AS loss_ratio
FROM premium pr
FULL OUTER JOIN paid pd
  ON pr.plan_code = pd.plan_code AND pr.month_start = pd.month_start
ORDER BY plan_code, month_start;

SELECT * FROM claims.analytics.w01 ORDER BY plan_code, month_start;

In [0]:
%sql
-- w02 materialized as claims.analytics.w02 (rolling 3-month loss ratio — ONE ratio over the window)
CREATE OR REPLACE TABLE claims.analytics.w02 AS
WITH w01 AS (
  -- identical CTEs to the w01 cell
  WITH months AS (
    SELECT DISTINCT
      month_start,
      ADD_MONTHS(month_start, 1) - INTERVAL 1 DAY AS month_end
    FROM claims.analytics.dim_date
  ),
  paid AS (
    SELECT p.plan_code, d.month_start, SUM(f.approved_amount) AS paid_amount
    FROM claims.analytics.fact_claims f
    JOIN claims.analytics.dim_plan p ON f.plan_sk = p.plan_sk
    JOIN claims.analytics.dim_date d ON f.settled_date_key = d.date_key
    WHERE f.status = 'settled'
    GROUP BY p.plan_code, d.month_start
  ),
  premium AS (
    SELECT p.plan_code, m.month_start, SUM(p.annual_premium) / 12.0 AS earned_premium
    FROM claims.analytics.dim_plan p
    JOIN months m
      ON p.valid_from <= m.month_end
     AND (p.valid_to IS NULL OR p.valid_to >= m.month_start)
    GROUP BY p.plan_code, m.month_start
  )
  SELECT
    COALESCE(pr.plan_code, pd.plan_code) AS plan_code,
    COALESCE(pr.month_start, pd.month_start) AS month_start,
    COALESCE(pd.paid_amount, 0)    AS paid_amount,
    COALESCE(pr.earned_premium, 0) AS earned_premium
  FROM premium pr
  FULL OUTER JOIN paid pd
    ON pr.plan_code = pd.plan_code AND pr.month_start = pd.month_start
)
SELECT
  plan_code,
  month_start,
  ROUND(paid_amount / NULLIF(earned_premium, 0), 2) AS loss_ratio,
  ROUND(
    SUM(paid_amount)     OVER (PARTITION BY plan_code ORDER BY month_start ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
    / NULLIF(SUM(earned_premium) OVER (PARTITION BY plan_code ORDER BY month_start ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 0),
  2) AS rolling_3m_loss_ratio
FROM w01
ORDER BY plan_code, month_start;

SELECT * FROM claims.analytics.w02 ORDER BY plan_code, month_start;

In [0]:
%sql
-- w03 materialized as claims.analytics.w03 — genuinely tied hospitals SHARE a rank (RANK)
CREATE OR REPLACE TABLE claims.analytics.w03 AS
SELECT
  h.hospital_id,
  h.city_tier,
  ROUND(SUM(f.approved_amount), 2) AS total_paid,
  RANK() OVER (PARTITION BY h.city_tier ORDER BY SUM(f.approved_amount) DESC) AS rank_in_tier
FROM claims.analytics.fact_claims f
JOIN claims.analytics.dim_hospital h ON f.hospital_sk = h.hospital_sk
WHERE f.status = 'settled'
GROUP BY h.hospital_id, h.city_tier
ORDER BY city_tier, rank_in_tier;

SELECT * FROM claims.analytics.w03 ORDER BY city_tier, rank_in_tier;

In [0]:
%sql
-- w04 materialized as claims.analytics.w04 — cohorts by policy start quarter
CREATE OR REPLACE TABLE claims.analytics.w04 AS
WITH cohorts AS (
  SELECT
    member_id,
    CONCAT(CAST(YEAR(MIN(valid_from)) AS STRING), '-Q', CAST(QUARTER(MIN(valid_from)) AS STRING)) AS cohort_quarter
  FROM claims.analytics.dim_plan
  GROUP BY member_id
),
per_cohort AS (
  SELECT
    c.cohort_quarter,
    COUNT(DISTINCT c.member_id) AS members,
    COUNT(DISTINCT f.member_sk) AS claimants,
    COUNT(f.claim_id)           AS claim_count
  FROM cohorts c
  LEFT JOIN claims.analytics.dim_member dm ON c.member_id = dm.member_id
  LEFT JOIN claims.analytics.fact_claims f ON dm.member_sk = f.member_sk
  GROUP BY c.cohort_quarter
)
SELECT
  cohort_quarter,
  members,
  claimants,
  claim_count,
  ROUND(claim_count / members, 4) AS claims_per_member,
  SUM(members) OVER (ORDER BY cohort_quarter ROWS UNBOUNDED PRECEDING) AS cumulative_members
FROM per_cohort
ORDER BY cohort_quarter;

SELECT * FROM claims.analytics.w04 ORDER BY cohort_quarter;